# Lab 10-04: Search Pattern Demonstrations

**Notebook 2 of 4.** Demonstrates six retrieval patterns against the `arxiv-nlp` index
created by `10-02-index-and-ingest.ipynb`. Run that notebook first.

**Why this matters** — Foundry IQ's agentic retrieval automatically selects and combines
these patterns for each sub-query. Understanding what happens under the hood explains
why KB retrieval outperforms single-query RAG by up to 40%.

**Patterns demonstrated:**

| # | Pattern | Mechanism |
|---|---------|----------|
| 1 | BM25 keyword search | Term frequency / inverse document frequency (lexical) |
| 2 | Pure vector search | Cosine similarity via integrated vectorizer (semantic) |
| 3 | Hybrid search | BM25 + vector fused with Reciprocal Rank Fusion |
| 4 | Semantic reranker | Hybrid + Azure AI language model re-scores candidates |
| 5 | OData-filtered hybrid | Structured pre-filter (e.g. year range) before ranking |
| 6 | Security-trimmed search | `group_ids` OData filter — document-level access control |

## Prerequisites

1. **Run `10-02-index-and-ingest.ipynb`** — the `arxiv-nlp` index must exist and be populated.
2. **`.env` file** — `IQ_SEARCH_ENDPOINT` must be set.
3. **Python environment** — run `uv sync`, select the `.venv` kernel.
4. **Azure CLI** — run `az login`.
5. **RBAC** — your identity needs **Search Index Data Reader** on the AI Search service.

## 1. Imports and configuration

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery
from display_helpers import show_search_results

repo_root = Path(__file__).resolve().parents[1] if '__file__' in dir() else Path.cwd().parent
load_dotenv(repo_root / '.env', override=True)

INDEX_NAME      = 'arxiv-nlp'
search_endpoint = os.environ['IQ_SEARCH_ENDPOINT']

print(f'Search endpoint: {search_endpoint}')
print(f'Index name     : {INDEX_NAME}')

Search endpoint: https://iq-search-gvwiex.search.windows.net
Index name     : arxiv-nlp


## 2. Create search client

`DefaultAzureCredential` uses the RBAC-assigned **Search Index Data Reader** role.
No admin key is needed.

In [2]:
credential = DefaultAzureCredential()

search_client = SearchClient(
    endpoint=search_endpoint,
    index_name=INDEX_NAME,
    credential=credential,
)

print(f"Search client: ready (index='{INDEX_NAME}')")

Search client: ready (index='arxiv-nlp')


## Pattern 1 — BM25 keyword search

Traditional full-text search on `title` and `abstract` fields. Results are ranked by
BM25 (term frequency × inverse document frequency). No vector computation.

**Strength**: exact term matches, known vocabulary, fast.<br>
**Weakness**: vocabulary mismatch — synonyms and paraphrases are missed.

In [3]:
QUERY = 'transformer models for text summarization'

results = list(search_client.search(
    search_text=QUERY,
    select=['title', 'year', 'abstract'],
    top=5,
))

show_search_results(QUERY, results, mode='BM25 keyword')

**Query (BM25 keyword):** *"transformer models for text summarization"*

#,Score,Year,Title,Snippet
1,15.1755,2016,Improving Multi-Document Summarization via Text Classification,"Developed so far, multi-document summarization has reached its bottleneck due to the lack of sufficient training data and diverse categories of documents. Text classification just makes up for these..."
2,15.1461,2016,Abstractive Text Summarization Using Sequence-to-Sequence RNNs and Beyond,"In this work, we model abstractive text summarization using Attentional Encoder-Decoder Recurrent Neural Networks, and show that they achieve state-of-the-art performance on two different corpora. W..."
3,15.0330,2016,LCSTS: A Large Scale Chinese Short Text Summarization Dataset,"Automatic text summarization is widely regarded as the highly difficult problem, partially because of the lack of large text summarization data set. Due to the great challenge of constructing the la..."
4,14.9840,2013,New Alignment Methods for Discriminative Book Summarization,"We consider the unsupervised alignment of the full text of a book with a human-written summary. This presents challenges not seen in other text alignment problems, including a disparity in length an..."
5,14.5356,2015,A Neural Attention Model for Abstractive Sentence Summarization,"Summarization based on text extraction is inherently limited, but generation-style abstractive methods have proven challenging to build. In this work, we propose a fully data-driven approach to abst..."


## Pattern 2 — Pure vector search

`VectorizableTextQuery` sends the query text to the integrated vectorizer, which calls
`text-embedding-3-large` via APIM at search time. Results are ranked by cosine
similarity against `abstractVector`.

**Strength**: semantic similarity — finds relevant papers even when they use different
terminology (e.g. "sequence-to-sequence" instead of "transformer").<br>
**Weakness**: may miss papers with exact keyword matches that have dissimilar embeddings.

In [4]:
results = list(search_client.search(
    search_text=None,
    vector_queries=[
        VectorizableTextQuery(
            text=QUERY,
            fields='abstractVector',
        )
    ],
    select=['title', 'year', 'abstract'],
    top=5,
))

show_search_results(QUERY, results, mode='pure vector')

**Query (pure vector):** *"transformer models for text summarization"*

#,Score,Year,Title,Snippet
1,0.7123,2016,Distraction-Based Neural Networks for Document Summarization,"Distributed representation learned with neural networks has recently shown to be effective in modeling natural languages at fine granularities such as words, phrases, and even sentences. Whether and..."
2,0.7118,2016,Abstractive Text Summarization Using Sequence-to-Sequence RNNs and Beyond,"In this work, we model abstractive text summarization using Attentional Encoder-Decoder Recurrent Neural Networks, and show that they achieve state-of-the-art performance on two different corpora. W..."
3,0.7090,2017,Text Summarization using Deep Learning and Ridge Regression,"We develop models and extract relevant features for automatic text summarization and investigate the performance of different models on the DUC 2001 dataset. Two different models were developed, one..."
4,0.7063,2016,Neural Summarization by Extracting Sentences and Words,Traditional approaches to extractive summarization rely heavily on human-engineered features. In this work we propose a data-driven approach based on neural networks and continuous sentence features...
5,0.7029,2005,Induction of Word and Phrase Alignments for Automatic Document Summarization,"Current research in automatic single document summarization is dominated by two effective, yet naive approaches: summarization by sentence extraction, and headline generation via bag-of-words models..."


## Pattern 3 — Hybrid search (BM25 + vector, RRF)

Both the BM25 and vector ranked lists are computed, then merged using
**Reciprocal Rank Fusion (RRF)**. A document scores well if it ranks highly in
either list. Hybrid typically outperforms either signal alone.

**Strength**: catches both exact-term hits and semantic near-misses.<br>
**Note**: RRF scores are normalised (0–1 range), so they are not comparable to
raw BM25 or cosine scores.

In [5]:
results = list(search_client.search(
    search_text=QUERY,
    vector_queries=[
        VectorizableTextQuery(
            text=QUERY,
            fields='abstractVector',
        )
    ],
    select=['title', 'year', 'abstract'],
    top=5,
))

show_search_results(QUERY, results, mode='hybrid BM25 + vector RRF')

**Query (hybrid BM25 + vector RRF):** *"transformer models for text summarization"*

#,Score,Year,Title,Snippet
1,0.0328,2016,Abstractive Text Summarization Using Sequence-to-Sequence RNNs and Beyond,"In this work, we model abstractive text summarization using Attentional Encoder-Decoder Recurrent Neural Networks, and show that they achieve state-of-the-art performance on two different corpora. W..."
2,0.0315,2017,Text Summarization using Deep Learning and Ridge Regression,"We develop models and extract relevant features for automatic text summarization and investigate the performance of different models on the DUC 2001 dataset. Two different models were developed, one..."
3,0.0314,2016,Distraction-Based Neural Networks for Document Summarization,"Distributed representation learned with neural networks has recently shown to be effective in modeling natural languages at fine granularities such as words, phrases, and even sentences. Whether and..."
4,0.0298,2016,Neural Summarization by Extracting Sentences and Words,Traditional approaches to extractive summarization rely heavily on human-engineered features. In this work we propose a data-driven approach based on neural networks and continuous sentence features...
5,0.0295,2015,A Neural Attention Model for Abstractive Sentence Summarization,"Summarization based on text extraction is inherently limited, but generation-style abstractive methods have proven challenging to build. In this work, we propose a fully data-driven approach to abst..."


## Pattern 4 — Semantic reranker

Hybrid search followed by Azure AI Search's **semantic reranker** — a language
understanding model that re-scores the top candidates based on semantic relevance to
the query and returns extractive captions from the most relevant passages.

The semantic configuration `arxiv-nlp-semantic` tells the reranker which fields are
the title (`title`), primary content (`abstract`), and keywords (`categories`).

**Strength**: highest relevance quality; best for complex, multi-concept queries.<br>
**Trade-off**: adds latency and costs semantic ranker units.

In [6]:
results = list(search_client.search(
    search_text=QUERY,
    vector_queries=[
        VectorizableTextQuery(
            text=QUERY,
            fields='abstractVector',
        )
    ],
    query_type='semantic',
    semantic_configuration_name='arxiv-nlp-semantic',
    query_caption='extractive',
    select=['title', 'year', 'abstract'],
    top=5,
))

# Semantic results include captions — show_search_results handles @search.reranker_score
show_search_results(QUERY, results, mode='hybrid + semantic reranker')

**Query (hybrid + semantic reranker):** *"transformer models for text summarization"*

#,Score,Year,Title,Snippet
1,2.6040,2016,Topic Sensitive Neural Headline Generation,Neural models have recently been used in text summarization including headline generation. The model can be trained using a set of document-headline pairs. We s... Taking advantage of topic informatio...
2,2.5271,2016,Abstractive Text Summarization Using Sequence-to-Sequence RNNs and Beyond,"In this work, we model abstractive text summarization using Attentional Encoder-Decoder Recurrent Neural Networks, and show that they achieve state-of-the-art performance on two different corpora. We ..."
3,2.5087,2016,Towards Abstraction from Extraction: Multiple Timescale Gated Recurrent Unit for Summarization,"In this work, we introduce temporal hierarchies to the sequence to sequence (seq2seq) model to tackle the problem of abstractive summarization of scientific articles. The proposed Multiple Timescale m..."
4,2.4635,2014,"Modelling, Visualising and Summarising Documents with a Single Convolutional Neural Network","Our model is based on an extended Dynamic Convolution Neural Network, which learns convolution filters at both the sentence and document level, hierarchically learning to capture and compose ... Inspi..."
5,2.4516,2017,Text Summarization using Deep Learning and Ridge Regression,"We develop models and extract relevant features for automatic text summarization and investigate the performance of different models on the DUC 2001 dataset. Two different models were developed, one b..."


## Pattern 5 — OData-filtered hybrid search

Structured pre-filter applied **before** scoring — only documents matching the
filter are considered. Filters are exact boolean predicates on filterable fields.

The example below restricts to papers published 2020–2024, showing how temporal
context can be injected into retrieval without reranking the entire corpus.

In [7]:
YEAR_FILTER = 'year ge 2020 and year le 2024'

results = list(search_client.search(
    search_text=QUERY,
    vector_queries=[
        VectorizableTextQuery(
            text=QUERY,
            fields='abstractVector',
        )
    ],
    filter=YEAR_FILTER,
    select=['title', 'year', 'abstract'],
    top=5,
))

print(f'Filter: {YEAR_FILTER}')
show_search_results(QUERY, results, mode='hybrid + OData year filter (2020-2024)')

Filter: year ge 2020 and year le 2024


**Query (hybrid + OData year filter (2020-2024)):** *"transformer models for text summarization"*

#,Score,Year,Title,Snippet
1,0.0331,2020,Incorporating Language Level Information into Acoustic Models,"This paper proposed a class of novel Deep Recurrent Neural Networks which can incorporate language-level information into acoustic models. For simplicity, we named these networks Recurrent Deep Lang..."
2,0.0323,2024,Reading Comprehension using Entity-based Memory Network,"This paper introduces a novel neural network model for question answering, the \emph{entity-based memory network}. It enhances neural networks' ability of representing and calculating information ov..."
3,0.0313,2020,Enhanced LSTM for Natural Language Inference,"Reasoning and inference are central to human and artificial intelligence. Modeling inference in human language is very challenging. With the availability of large annotated data (Bowman et al., 2015..."
4,0.0311,2020,Median-Based Generation of Synthetic Speech Durations using a Non-Parametric Approach,This paper proposes a new approach to duration modelling for statistical parametric speech synthesis in which a recurrent statistical model is trained to output a phone transition probability at eac...
5,0.0310,2022,A Probabilistic Generative Grammar for Semantic Parsing,"Domain-general semantic parsing is a long-standing goal in natural language processing, where the semantic parser is capable of robustly parsing sentences from domains outside of which it was traine..."


## Pattern 6 — Security-trimmed search

Document-level access control using the `group_ids` Collection field set during
ingest. The OData expression `group_ids/any(g: search.in(g, '<groups>'))` filters
results to documents tagged with at least one of the caller's group memberships.

**This is applied at query time** — it cannot be bypassed by the caller because it
is enforced by the search service, not the application layer.

| Group | Documents visible |
|-------|------------------|
| `public` | Papers from 2010 onwards |
| `archive` | All papers (1994 onwards) |

> Notebook 10-03-knowledge-base-setup demonstrates the same pattern at the Foundry IQ knowledge base
> level via `filterAddOn` on knowledge base queries.

In [8]:
def security_filtered_search(query: str, user_groups: list, top: int = 5):
    """Hybrid search restricted to documents tagged with at least one user group."""
    groups_csv = ', '.join(user_groups)
    security_filter = f"group_ids/any(g: search.in(g, '{groups_csv}'))"

    results = list(search_client.search(
        search_text=query,
        vector_queries=[
            VectorizableTextQuery(text=query, fields='abstractVector')
        ],
        filter=security_filter,
        select=['title', 'year', 'abstract', 'group_ids'],
        top=top,
    ))
    return results, security_filter


# Caller has 'public' membership — sees only 2010+ papers
results_public, filt = security_filtered_search(QUERY, user_groups=['public'])
print(f'Filter applied: {filt}')
print(f'Results for public-only caller ({len(results_public)} docs):')
for doc in results_public:
    print(f'  [{doc["year"]}] {doc["title"]}')

print()

# Caller has 'archive' membership — sees all papers
results_archive, filt = security_filtered_search(QUERY, user_groups=['archive'])
print(f'Filter applied: {filt}')
print(f'Results for archive caller ({len(results_archive)} docs):')
for doc in results_archive:
    print(f'  [{doc["year"]}] {doc["title"]}')

Filter applied: group_ids/any(g: search.in(g, 'public'))
Results for public-only caller (5 docs):
  [2016] Abstractive Text Summarization Using Sequence-to-Sequence RNNs and
  Beyond
  [2017] Text Summarization using Deep Learning and Ridge Regression
  [2016] Distraction-Based Neural Networks for Document Summarization
  [2016] Neural Summarization by Extracting Sentences and Words
  [2015] A Neural Attention Model for Abstractive Sentence Summarization

Filter applied: group_ids/any(g: search.in(g, 'archive'))
Results for archive caller (5 docs):
  [2016] Abstractive Text Summarization Using Sequence-to-Sequence RNNs and
  Beyond
  [2017] Text Summarization using Deep Learning and Ridge Regression
  [2016] Distraction-Based Neural Networks for Document Summarization
  [2016] Neural Summarization by Extracting Sentences and Words
  [2015] A Neural Attention Model for Abstractive Sentence Summarization


## Summary

| Pattern | When to use |
|---------|------------|
| BM25 | Known vocabulary, exact-term queries, low latency |
| Vector | Conceptual queries, paraphrases, cross-lingual |
| Hybrid (RRF) | General purpose — best default choice |
| Semantic reranker | High-quality results needed; complex multi-concept queries |
| OData-filtered | Structured facets known at query time (date, category, status) |
| Security-trimmed | Multi-tenant, per-document access control |

Foundry IQ's agentic retrieval pipeline (10-03-knowledge-base-setup) automatically selects
from these patterns for each sub-query and semantically reranks all results before
returning them to the agent.